# Phase 8: generation run on Kaggle (alternative to Colab)

**One-time setup**
1. Kaggle account with a **verified phone number** (needed for GPU and internet).
2. Upload `rag_data.zip` (data/raw + data/processed + data/chroma) as a **private dataset**: *Datasets → New Dataset*.
   Kaggle unpacks the zip; this notebook finds the files wherever they land under `/kaggle/input/`.
3. In this notebook: *Add Input* → your dataset. *Settings → Accelerator → GPU T4 x2* (or P100). *Settings → Internet → On*.

**Run:** *Save Version → Save & Run All (Commit)*. It runs in the background (you can close the browser) and keeps
everything in `/kaggle/working`. When the version finishes, open it → **Output** → download `gen_llama.zip`.

Every cell stops on the first error, so a failure shows up at the step that caused it (see the version's **Logs**).
The qwen2.5:7b fallback is **not** run here: the last cell only reports whether its pre-registered trigger fired.

**Resume after a failed version:** *Add Input → Your Work →* this notebook's failed version (if it saved output).
Cell 5 then copies its `gen_llama/generations.jsonl` back and the run continues from the saved answers.

## 1. Settings and a shell helper that stops on errors

In [ ]:
import glob, os, shutil, subprocess, time, urllib.request

REPO = "/tmp/rag"                      # code + data live here (not saved as output)
OUT = "/kaggle/working/gen_llama"      # results: the only thing saved as output
BRANCH = "phase-8-generation"
RUN_CONFIG = "configs/gen_llama.yaml"

def sh(cmd):
    """Run a bash command, print its output as it comes, and stop the notebook if it fails."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(f"set -o pipefail; {cmd}", shell=True, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}")

## 2. Code from GitHub, data from the Kaggle dataset, packages (about 3-5 min)
Expect the latest commit line, then `Index chunks: 2468 (expect 2468)`.

In [ ]:
sh(f"rm -rf {REPO} && git clone -q -b {BRANCH} https://github.com/himalSubedi0602/Rag_System.git {REPO}")
os.chdir(REPO)
sh("git log --oneline -1")

found = glob.glob("/kaggle/input/**/data/chroma/chroma.sqlite3", recursive=True)
zips = glob.glob("/kaggle/input/**/rag_data.zip", recursive=True)
if found:                                   # Kaggle unpacked the zip: copy data/raw, processed, chroma
    root = os.path.dirname(os.path.dirname(os.path.dirname(found[0])))
    for sub in ("raw", "processed", "chroma"):
        shutil.copytree(f"{root}/data/{sub}", f"{REPO}/data/{sub}", dirs_exist_ok=True)
    print("Data copied from", root)
elif zips:                                  # the zip was kept as a file
    sh(f"unzip -q -o '{zips[0]}' -d {REPO}")
else:
    raise FileNotFoundError("No data found under /kaggle/input: add your rag_data dataset (Add Input).")

sh("pip install -q -r requirements.txt 2>&1 | grep -v -i 'dependency' | tail -5 || true")
import chromadb
n = chromadb.PersistentClient("data/chroma").get_collection("fixed-500-50__bge-small__noprefix").count()
print("Index chunks:", n, "(expect 2468)")
assert n == 2468, "wrong or missing Chroma index in the dataset"

## 3. Ollama and both models (about 4-6 min)
The last line lists `llama3.1:8b` and `qwen2.5:7b`.

In [ ]:
sh("apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null")
sh("curl -fsSL https://ollama.com/install.sh | sh")
subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):                         # wait until the server answers
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not start: see /tmp/ollama.log")
for model in ("llama3.1:8b", "qwen2.5:7b"):
    sh(f"ollama pull {model} 2>&1 | tail -1")
sh("ollama list")

## 4. Tests and a 5-question smoke run (about 2 min)
GPU for embeddings (`device: auto`; `cpu` is only for the 8 GB Mac). Expect `138 passed` or more, then 5 answers. The smoke run writes to `gen_llama_smoke` and is not a result.

In [ ]:
sh(f"sed -i 's/^device: cpu/device: auto/' {RUN_CONFIG} && grep '^device' {RUN_CONFIG}")
sh("python -m pytest -q 2>&1 | tail -2")
sh(f"python run_generation.py {RUN_CONFIG} --limit 5")

## 5. Results folder in /kaggle/working (resumes from an attached earlier output, if any)

In [ ]:
os.makedirs(OUT, exist_ok=True)
earlier = glob.glob("/kaggle/input/**/gen_llama/generations.jsonl", recursive=True)
if earlier and not os.path.exists(f"{OUT}/generations.jsonl"):
    src = os.path.dirname(earlier[0])
    for name in ("generations.jsonl", "config.yaml"):
        if os.path.exists(f"{src}/{name}"):
            shutil.copy(f"{src}/{name}", OUT)
    print("Resuming from", src)
else:
    print("Fresh run" if not earlier else "Answers already in the output folder")
sh(f"rm -rf eval/results/gen_llama && ln -s {OUT} eval/results/gen_llama && ls -la eval/results/gen_llama/")

## 6. Full tune run: 159 questions, then grading (about 20-30 min on a T4)

In [ ]:
sh(f"python run_generation.py {RUN_CONFIG}")
sh(f"ls -la {OUT}")

## 7. Zip for download, and report the fallback trigger (does NOT run the fallback)

In [ ]:
import pandas as pd
sh("cd /kaggle/working && rm -f gen_llama.zip && zip -r -q gen_llama.zip gen_llama && ls -lh gen_llama.zip")
m = pd.read_csv(f"{OUT}/metrics.csv", index_col=0).loc["overall"]
fmt, sup = m["cit_format"], m["cit_supported"]
print(f"Citation format {fmt:.3f} (trigger < 0.90), support {sup:.3f} (trigger < 0.70)")
print("FALLBACK TRIGGER FIRED: report it before running anything." if fmt < 0.90 or sup < 0.70
      else "No fallback trigger: llama3.1:8b stays.")
print(open(f"{OUT}/summary.md", encoding="utf-8").read())